## 1. Install dependencies


In [40]:
import sys
!{sys.executable} -m pip install -q "scikit-learn==1.6.1" vaderSentiment emoji pandas matplotlib joblib

## 2. Provide the dataset
Colab: a file picker appears — choose **`raw_reviews.csv`**.  
Local Jupyter: put `raw_reviews.csv` in a `data/` folder next to this notebook.


In [ ]:
import os
os.makedirs('data', exist_ok=True); os.makedirs('models', exist_ok=True)

path = '/content/data/translated_reviews_transformer_sentiment.csv' ##change to actual file path
os.replace(path, 'data/raw_reviews.csv')
print(f'Saved to data/raw_reviews.csv')

Saved to data/raw_reviews.csv


## 3. Write the shared modules
`preprocess.py` and `features.py` must exist as files so the saved model can be reloaded by the web app
(the model stores a reference to `features.VaderFeatures`). These are the **same** files the web app uses.


In [42]:
import re
import emoji

_ASPECT_RATING = re.compile(
    r"(food|service|atmosphere|ambien?ce)\s*:?\s*\d(?:\.\d)?\s*/\s*5",
    re.IGNORECASE,
)

_META_MARKERS = [
    r"Group size", r"Wait time", r"Seating type", r"Price per person",
    r"Reservation recommended", r"Service\s*:\s*Dine in", r"Meal type",
    r"Recommended dishes", r"Parking space", r"Parking options",
    r"Dietary options", r"Kid-friendliness", r"Wheelchair",
]
_META_RE = re.compile("|".join(_META_MARKERS), re.IGNORECASE)

_URL = re.compile(r"https?://\S+|www\.\S+")
_MULTISPACE = re.compile(r"\s+")
_NOISE_SYMBOLS = re.compile(r"[✓✔|•]+")


def strip_metadata(text: str) -> str:
    """Remove trailing Google Maps metadata and embedded aspect ratings."""
    if not isinstance(text, str):
        return ""
    text = _ASPECT_RATING.sub(" ", text)
    m = _META_RE.search(text)
    if m:
        text = text[: m.start()]
    return text


def clean_text(text: str, demojize: bool = True) -> str:
    if not isinstance(text, str):
        return ""
    text = strip_metadata(text)
    text = _URL.sub(" ", text)
    text = _NOISE_SYMBOLS.sub(" ", text)
    if demojize:
        text = emoji.demojize(text, delimiters=(" ", " "))
        text = text.replace("_", " ")
    text = text.lower()
    text = _MULTISPACE.sub(" ", text).strip()
    return text


def extract_emojis(text: str):
    if not isinstance(text, str):
        return []
    return [e["emoji"] for e in emoji.emoji_list(text)]


if __name__ == "__main__":
    sample = ("Food tasted good, staff also very attentive 😍 "
              "Food: 5/5 | Service: 5/5 | Atmosphere: 5/5 "
              "Group size 3-4 people Wait time Up to 10 min")
    print("RAW :", sample)
    print("META:", strip_metadata(sample))
    print("CLEAN:", clean_text(sample))
    print("EMOJI:", extract_emojis(sample))


RAW : Food tasted good, staff also very attentive 😍 Food: 5/5 | Service: 5/5 | Atmosphere: 5/5 Group size 3-4 people Wait time Up to 10 min
META: Food tasted good, staff also very attentive 😍   |   |   
CLEAN: food tasted good, staff also very attentive smiling face with heart-eyes
EMOJI: ['😍']


In [43]:
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer


class VaderFeatures(BaseEstimator, TransformerMixin):

    def __init__(self):
        self._analyzer = None

    @property
    def analyzer(self):
        # lazy so the object pickles cleanly
        if self._analyzer is None:
            self._analyzer = SentimentIntensityAnalyzer()
        return self._analyzer

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        rows = []
        for t in X:
            s = self.analyzer.polarity_scores(t if isinstance(t, str) else "")
            rows.append([s["neg"], s["neu"], s["pos"], s["compound"]])
        return np.asarray(rows, dtype=float)

    def __getstate__(self):
        return {}

    def __setstate__(self, state):
        self._analyzer = None


## 4. Training + analytics scripts
Written to disk and run with `!python` so file paths resolve correctly.


In [44]:
import json
import os
import warnings

import joblib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import ComplementNB
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold

from features import VaderFeatures
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, confusion_matrix,
)

from preprocess import clean_text

warnings.filterwarnings("ignore")

HERE = os.getcwd()
RAW_CSV = os.path.join(HERE, "data", "raw_reviews.csv")
MODEL_DIR = os.path.join(HERE, "models")
DATA_DIR = os.path.join(HERE, "data")
LABELS = ["negative", "neutral", "positive"]
SEED = 42


def load_data():
    df = pd.read_csv(RAW_CSV)
    base = df["Review_clean"].fillna(df["Review_en"]).fillna(df["Review"])
    df["text_clean"] = base.apply(clean_text)
    df = df[df["text_clean"].str.len() > 0].copy()
    df = df[df["sentiment"].isin(LABELS)].copy()
    return df


def build_models():
    def tfidf_union():
        return [
            ("word", TfidfVectorizer(
                ngram_range=(1, 2), min_df=2, max_df=0.9,
                sublinear_tf=True, strip_accents="unicode")),
            ("char", TfidfVectorizer(
                analyzer="char_wb", ngram_range=(3, 5), min_df=2,
                sublinear_tf=True)),
        ]

    def hybrid_feats():
        return FeatureUnion(tfidf_union() + [
            ("vader", Pipeline([("v", VaderFeatures()),
                                ("scale", StandardScaler())])),
        ])

    return {
        "logreg": Pipeline([
            ("feats", hybrid_feats()),
            ("clf", LogisticRegression(
                max_iter=3000, class_weight="balanced", C=2.0, random_state=SEED)),
        ]),
        "linearsvc": Pipeline([
            ("feats", hybrid_feats()),
            ("clf", LinearSVC(class_weight="balanced", C=1.0, random_state=SEED)),
        ]),
        "complementnb": Pipeline([
            ("feats", FeatureUnion(tfidf_union())),
            ("clf", ComplementNB()),
        ]),
    }


def main():
    os.makedirs(MODEL_DIR, exist_ok=True)
    df = load_data()
    print(f"Loaded {len(df)} usable reviews")
    print("Label distribution:\n", df["sentiment"].value_counts().to_string())

    X, y = df["text_clean"].values, df["sentiment"].values
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=SEED)

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    results, fitted = {}, {}

    for name, pipe in build_models().items():
        cv_f1 = cross_val_score(pipe, X_tr, y_tr, cv=cv,
                                scoring="f1_macro").mean()
        pipe.fit(X_tr, y_tr)
        pred = pipe.predict(X_te)
        acc = accuracy_score(y_te, pred)
        f1m = f1_score(y_te, pred, average="macro")
        f1w = f1_score(y_te, pred, average="weighted")
        results[name] = {
            "cv_macro_f1": round(float(cv_f1), 4),
            "test_accuracy": round(float(acc), 4),
            "test_macro_f1": round(float(f1m), 4),
            "test_weighted_f1": round(float(f1w), 4),
            "per_class": classification_report(
                y_te, pred, labels=LABELS, output_dict=True, zero_division=0),
        }
        fitted[name] = pipe
        print(f"  {name:<13} acc={acc:.3f}  macroF1={f1m:.3f}  cvF1={cv_f1:.3f}")

    best = max(results, key=lambda k: results[k]["cv_macro_f1"])
    print(f"\nBest model (by CV macro-F1): {best}")

    # refit best on ALL data so the deployed model uses every labelled example
    final = build_models()[best].fit(X, y)
    joblib.dump(final, os.path.join(MODEL_DIR, "sentiment_model.joblib"))

    # confusion matrix on the held-out test set (for the report figure)
    cm = confusion_matrix(y_te, fitted[best].predict(X_te), labels=LABELS)
    fig, ax = plt.subplots(figsize=(4.6, 4.2))
    im = ax.imshow(cm, cmap="YlOrRd")
    ax.set_xticks(range(3)); ax.set_yticks(range(3))
    ax.set_xticklabels(LABELS); ax.set_yticklabels(LABELS)
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
    ax.set_title(f"Confusion Matrix — {best}")
    for i in range(3):
        for j in range(3):
            ax.text(j, i, cm[i, j], ha="center", va="center",
                    color="black", fontweight="bold")
    fig.colorbar(im, fraction=0.046, pad=0.04)
    fig.tight_layout()
    fig.savefig(os.path.join(MODEL_DIR, "confusion_matrix.png"), dpi=130)

    meta = {
        "best_model": best,
        "labels": LABELS,
        "n_train": len(X_tr), "n_test": len(X_te), "n_total": len(X),
        "train_label_counts": pd.Series(y_tr).value_counts().to_dict(),
        "results": results,
    }
    with open(os.path.join(MODEL_DIR, "metrics.json"), "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2)

    # knowledge representation: cleaned, model-ready dataset
    df[["Name", "Rating", "Review_en", "text_clean",
        "food_rating", "service_rating", "atmosphere_rating",
        "sentiment"]].to_csv(
        os.path.join(DATA_DIR, "reviews_clean.csv"), index=False)

    print("Saved model, metrics, confusion matrix, cleaned dataset.")


if __name__ == "__main__":
    main()


Loaded 377 usable reviews
Label distribution:
 sentiment
positive    269
negative     83
neutral      25
  logreg        acc=0.816  macroF1=0.643  cvF1=0.640
  linearsvc     acc=0.803  macroF1=0.638  cvF1=0.614
  complementnb  acc=0.671  macroF1=0.423  cvF1=0.553

Best model (by CV macro-F1): logreg
Saved model, metrics, confusion matrix, cleaned dataset.


In [45]:
import json
import os
import re
from collections import Counter

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

from preprocess import clean_text, extract_emojis

HERE = os.getcwd()
DATA_DIR = os.path.join(HERE, "data")
RAW_CSV = os.path.join(DATA_DIR, "raw_reviews.csv")

ASPECTS = {
    "food": [
        "food", "dish", "dishes", "taste", "tasty", "delicious", "yummy",
        "flavour", "flavor", "flavourful", "meal", "menu", "portion", "fresh",
        "cook", "cooked", "spicy", "sweet", "sour", "rice", "noodle", "noodles",
        "chicken", "fish", "soup", "coffee", "drink", "drinks", "dessert",
        "bread", "cheese", "sauce", "fried", "grill", "grilled", "serving",
        "ingredient", "quality", "hot", "cold", "bland", "oily", "salty",
    ],
    "service": [
        "service", "staff", "staffs", "waiter", "waitress", "server", "crew",
        "friendly", "attentive", "helpful", "rude", "slow", "wait", "waiting",
        "attitude", "manager", "order", "ordered", "polite", "smile", "smiling",
        "ignore", "ignored", "unfriendly", "efficient", "prompt", "professional",
        "customer", "welcoming", "responsive",
    ],
    "ambiance": [
        "ambience", "ambiance", "atmosphere", "environment", "vibe", "vibes",
        "cozy", "cosy", "comfortable", "comfy", "decor", "decoration", "music",
        "clean", "cleanliness", "dirty", "seating", "seat", "crowded", "noisy",
        "quiet", "spacious", "aircon", "air-cond", "lighting", "interior",
        "relaxing", "warm", "cool", "modern", "aesthetic", "instagrammable",
    ],
}

ASPECTS["price"] = [
    "price", "prices", "pricey", "expensive", "cheap", "affordable", "value",
    "worth", "reasonable", "overpriced", "cost", "rm", "ringgit", "budget",
]

STOP = set("""
a an the and or but if then else for to of in on at by with from into about as
is are was were be been being do does did doing have has had having i you he she
it we they me him her us them my your his its our their this that these those
not no so very just too also only out up down off over again more most some any
all can will would could should may might must here there what which who whom
when where why how than then once their our we you i me my place here go went
will been im its u ur also got get really quite will visit came come back again
""".split())


def load():
    df = pd.read_csv(RAW_CSV)
    base = df["Review_clean"].fillna(df["Review_en"]).fillna(df["Review"])
    df["text_clean"] = base.apply(clean_text)
    df["text_raw"] = base.fillna("")
    # numeric star rating from "Rated 5.0 out of 5"
    df["stars"] = df["Rating"].astype(str).str.extract(r"([0-5](?:\.\d)?)").astype(float)
    return df


def tokens(text):
    return [t for t in re.findall(r"[a-z]+", str(text).lower())
            if len(t) > 2 and t not in STOP]


def log_odds_keywords(texts_a, texts_b, top_n=12):
    ca, cb = Counter(), Counter()
    for t in texts_a:
        ca.update(tokens(t))
    for t in texts_b:
        cb.update(tokens(t))
    vocab = set(ca) | set(cb)
    a_tot, b_tot = sum(ca.values()), sum(cb.values())
    a0 = a_tot + len(vocab)
    b0 = b_tot + len(vocab)
    scores = {}
    for w in vocab:
        ya, yb = ca[w], cb[w]
        if ya + yb < 2:          # ignore ultra-rare words
            continue
        la = np.log((ya + 1) / (a0 - ya - 1))
        lb = np.log((yb + 1) / (b0 - yb - 1))
        delta = la - lb
        var = 1.0 / (ya + 1) + 1.0 / (yb + 1)
        scores[w] = delta / np.sqrt(var)
    ranked = sorted(scores.items(), key=lambda kv: kv[1], reverse=True)
    return [(w, round(float(s), 2)) for w, s in ranked[:top_n]]


def freq_keywords(texts, top_n=12):
    c = Counter()
    for t in texts:
        c.update(set(tokens(t)))   # set -> count reviews mentioning the word
    return [(w, n) for w, n in c.most_common(top_n)]


def aspect_breakdown(df):
    out = {}
    for aspect, lex in ASPECTS.items():
        lexset = set(lex)
        mask = df["text_clean"].apply(lambda t: bool(lexset & set(tokens(t))))
        sub = df[mask]
        if len(sub) == 0:
            continue
        sent = sub["sentiment"].value_counts()
        pos = int(sent.get("positive", 0))
        neu = int(sent.get("neutral", 0))
        neg = int(sent.get("negative", 0))
        tot = pos + neu + neg
        # example snippets: one positive, one negative
        ex_pos = sub[sub["sentiment"] == "positive"]["Review_en"].head(1).tolist()
        ex_neg = sub[sub["sentiment"] == "negative"]["Review_en"].head(1).tolist()
        out[aspect] = {
            "mentions": int(len(sub)),
            "positive": pos, "neutral": neu, "negative": neg,
            "pos_pct": round(100 * pos / tot, 1) if tot else 0,
            "neg_pct": round(100 * neg / tot, 1) if tot else 0,
            "avg_stars": round(float(sub["stars"].mean()), 2) if sub["stars"].notna().any() else None,
            "example_positive": _short(ex_pos[0]) if ex_pos else None,
            "example_negative": _short(ex_neg[0]) if ex_neg else None,
            "top_words": [w for w, _ in freq_keywords(sub["text_clean"], 8)],
        }
    return out


def _short(text, n=180):
    text = re.sub(r"\s+", " ", str(text)).strip()
    return text[:n] + ("…" if len(text) > n else "")


def main():
    df = load()
    n = len(df)
    sent = df["sentiment"].value_counts()
    pos, neu, neg = (int(sent.get(k, 0)) for k in ["positive", "neutral", "negative"])

    pos_txt = df[df["sentiment"] == "positive"]["text_clean"]
    neg_txt = df[df["sentiment"] == "negative"]["text_clean"]
    other_txt = df[df["sentiment"] != "positive"]["text_clean"]
    nonneg_txt = df[df["sentiment"] != "negative"]["text_clean"]

    aspects = aspect_breakdown(df)

    # star-rating groups
    five = df[df["stars"] == 5.0]["text_clean"]
    one = df[df["stars"] <= 2.0]["text_clean"]   # 1-2 stars = "low star"
    rest_hi = df[df["stars"] < 5.0]["text_clean"]
    rest_lo = df[df["stars"] > 2.0]["text_clean"]

    # aspect rating averages (from the dataset's numeric columns)
    aspect_ratings = {}
    for col, key in [("food_rating", "food"), ("service_rating", "service"),
                     ("atmosphere_rating", "atmosphere")]:
        vals = df[col].dropna()
        aspect_ratings[key] = {
            "avg": round(float(vals.mean()), 2) if len(vals) else None,
            "n": int(len(vals)),
        }

    # emoji tally
    emoji_counter = Counter()
    for t in df["text_raw"]:
        emoji_counter.update(extract_emojis(t))
    top_emojis = [{"emoji": e, "count": int(c)} for e, c in emoji_counter.most_common(10)]

    analytics = {
        "n_reviews": n,
        "overall": {
            "positive": pos, "neutral": neu, "negative": neg,
            "pos_pct": round(100 * pos / n, 1),
            "neu_pct": round(100 * neu / n, 1),
            "neg_pct": round(100 * neg / n, 1),
            "avg_stars": round(float(df["stars"].mean()), 2),
        },
        "star_distribution": {
            str(int(s)): int((df["stars"] == s).sum())
            for s in sorted(df["stars"].dropna().unique())
        },
        "aspects": aspects,
        "aspect_ratings": aspect_ratings,
        "top_emojis": top_emojis,
        "questions": {
            "like_most": {
                "title": "What do customers like most?",
                "keywords": log_odds_keywords(pos_txt, other_txt, 12),
                "summary": _summary_like(aspects),
            },
            "complaints": {
                "title": "What are the most common complaints?",
                "keywords": log_odds_keywords(neg_txt, nonneg_txt, 12),
                "summary": _summary_complaints(aspects),
            },
            "negative_themes": {
                "title": "What themes appear in negative reviews?",
                "keywords": freq_keywords(neg_txt, 12),
                "examples": [_short(x) for x in
                             df[df["sentiment"] == "negative"]["Review_en"].head(3)],
            },
            "ambience": {
                "title": "How do customers describe the ambience?",
                "aspect": aspects.get("ambiance", {}),
            },
            "overall_summary": {
                "title": "Overall customer sentiment",
                "line": (f"{round(100*pos/n)}% positive, {round(100*neu/n)}% neutral, "
                         f"{round(100*neg/n)}% negative across {n} reviews "
                         f"(avg {round(float(df['stars'].mean()),1)}\u2605)."),
            },
            "service_vs_food": {
                "title": "Why is the service rating lower than the food rating?",
                "food_avg": aspect_ratings["food"]["avg"],
                "service_avg": aspect_ratings["service"]["avg"],
                "atmosphere_avg": aspect_ratings["atmosphere"]["avg"],
                "food_neg_pct": aspects.get("food", {}).get("neg_pct"),
                "service_neg_pct": aspects.get("service", {}).get("neg_pct"),
                "service_complaint_words": log_odds_keywords(
                    df[(df["sentiment"] == "negative")]["text_clean"]
                    .pipe(lambda s: s[s.apply(lambda t: bool(set(ASPECTS["service"]) & set(tokens(t))))]),
                    df[(df["sentiment"] == "positive")]["text_clean"], 8),
            },
            "five_star_drivers": {
                "title": "What factors drive 5-star reviews?",
                "keywords": log_odds_keywords(five, rest_hi, 12),
                "n": int((df["stars"] == 5.0).sum()),
            },
            "one_star_issues": {
                "title": "What issues appear in 1-star reviews?",
                "keywords": log_odds_keywords(one, rest_lo, 12),
                "n": int((df["stars"] <= 2.0).sum()),
                "examples": [_short(x) for x in
                             df[df["stars"] <= 2.0]["Review_en"].head(3)],
            },
        },
    }

    with open(os.path.join(DATA_DIR, "analytics.json"), "w", encoding="utf-8") as f:
        json.dump(analytics, f, indent=2, ensure_ascii=False)
    print(f"Wrote analytics.json  ({n} reviews; "
          f"{pos} pos / {neu} neu / {neg} neg)")


def _summary_like(aspects):
    ranked = sorted(aspects.items(), key=lambda kv: kv[1]["pos_pct"], reverse=True)
    top = [a for a, d in ranked if d["mentions"] >= 10][:2]
    return ("Most appreciated aspects: " + ", ".join(top) + ".") if top else ""


def _summary_complaints(aspects):
    ranked = sorted(aspects.items(), key=lambda kv: kv[1]["neg_pct"], reverse=True)
    top = [a for a, d in ranked if d["mentions"] >= 10][:2]
    return ("Most complained-about aspects: " + ", ".join(top) + ".") if top else ""


if __name__ == "__main__":
    main()


Wrote analytics.json  (419 reviews; 300 pos / 36 neu / 83 neg)


## 5. Check the results


In [46]:
import json, pandas as pd
m = json.load(open('models/metrics.json', encoding='utf-8'))
print('Best model:', m['best_model'])
pd.DataFrame([{'model':k,'cv_macro_f1':v['cv_macro_f1'],
               'test_accuracy':v['test_accuracy'],'test_macro_f1':v['test_macro_f1']}
              for k,v in m['results'].items()]).set_index('model')

Best model: logreg


,cv_macro_f1,test_accuracy,test_macro_f1
model,,,
logreg,0.6396,0.8158,0.6426
linearsvc,0.6144,0.8026,0.6377
complementnb,0.5531,0.6711,0.4228


In [47]:
import joblib, features
from preprocess import clean_text
clf = joblib.load('models/sentiment_model.joblib')
for t in ['Food sedap, staff friendly, will come again!',
          'Service damn slow and the waiter was rude',
          'Ambience nice but a bit pricey lah']:
    print(clf.predict([clean_text(t)])[0], '|', t)

positive | Food sedap, staff friendly, will come again!
negative | Service damn slow and the waiter was rude
negative | Ambience nice but a bit pricey lah


## 6. Export the trained files for the web app
Bundles only the three files the app needs and downloads them.


In [48]:
import zipfile, os
need = ['models/sentiment_model.joblib','models/metrics.json','data/analytics.json']
with zipfile.ZipFile('trained_tnl.zip','w',zipfile.ZIP_DEFLATED) as z:
    for p in need:
        if os.path.exists(p): z.write(p); print('added', p)
        else: print('MISSING', p)
print('Wrote trained_artifacts.zip')
try:
    from google.colab import files
    files.download('trained_tnl.zip')
except ImportError:
    print('Not in Colab — find trained_tnl.zip in the file browser.')

added models/sentiment_model.joblib
added models/metrics.json
added data/analytics.json
Wrote trained_artifacts.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Put the files into the web app
Unzip `trained_tnl.zip` and copy:
- `sentiment_model.joblib` and `metrics.json` → the web app's **`models/`**
- `analytics.json` → the web app's **`data/`**

Then run the web app with `python app.py`.

Run `pip install -r requirements.txt` first to install package.
